In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark import pipelines as dp

In [0]:

CATALOG = spark.conf.get("catalog")
SILVER_SCHEMA = spark.conf.get("silver_schema")
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.silver_view_events"

GOLD_SCHEMA = spark.conf.get("gold_schema")

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.dim_show")
def dim_show():
    return(
        spark.read.table(SILVER_TABLE)
        .select("show_id", "title", "type", "rating", "release_year", "audience_category", "release_period")
        .dropDuplicates(["show_id"])
        .withColumn(
            "show_key", F.row_number().over(Window.orderBy("show_id"))
        )
        .select(
            "show_key", 
            "show_id", 
            "title", 
            "type", 
            "rating", 
            "release_year", 
            "audience_category", 
            "release_period"
        )
    )
    

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.dim_date")
def dim_date():
    df = spark.read.table(SILVER_TABLE)
    return(
        df.select("view_date")
        .filter(F.col("view_date").isNotNull())
        .distinct()
        .withColumn(
            "date_key", F.date_format("view_date", "yyyyMMdd").cast("int")
        )
        .withColumn("year", F.year("view_date"))
        .withColumn("month", F.month("view_date"))
        .withColumn("day", F.dayofmonth("view_date"))
        .withColumn("day_of_week", F.dayofweek("view_date"))
        .select(
            "date_key", 
            "view_date", 
            "year", 
            "month", 
            "day", 
            "day_of_week"
        )
    )

In [0]:
@dp.materialized_view(name=f"{GOLD_SCHEMA}.dim_user")
def dim_user():
    df = spark.read.table(SILVER_TABLE)
    return(
        df.select("user_id")
        .filter(F.col("user_id").isNotNull())
        .distinct()
        .withColumn("user_key", F.row_number().over(Window.orderBy("user_id")))
        .select("user_key", "user_id")
    )